In [3]:
from pathlib import Path
import duckdb
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

data_dir = Path("../data").resolve()
assert data_dir.is_dir(), f"Data folder not found: {data_dir}"

con = duckdb.connect(str(data_dir / "project.duckdb"))

print("Database:", data_dir / "project.duckdb")
print(con.sql("SELECT version()").fetchall()[0][0])

Database: C:\Users\manth\DATA4600 Final Project\causal-inference-project\data\project.duckdb
v2.0.0


## Clean Incident Table

In [4]:
con.sql("""
    CREATE OR REPLACE TABLE incident_clean AS
    SELECT
        try_cast(incident_id AS BIGINT) AS incident_id,
        try_cast(agency_id AS BIGINT) AS agency_id,
        source_state,
        source_year,
        coalesce(
            try_strptime(incident_date, '%Y-%m-%d'),
            try_strptime(incident_date, '%d-%b-%y')
        )::DATE AS incident_date,
        try_cast(incident_hour AS INTEGER) AS incident_hour
    FROM incident
""")

raw_rows = con.sql("SELECT count(*) FROM incident").fetchall()[0][0]
clean_rows = con.sql("SELECT count(*) FROM incident_clean").fetchall()[0][0]
assert raw_rows == clean_rows, f"incident: raw {raw_rows}, clean {clean_rows}"
print(f"incident_clean: {clean_rows} rows")

incident_clean: 65448378 rows


## Verify

In [ ]:
# NULLs before vs after cleaning, per column and year.
# failed = clean NULLs - raw NULLs = values that existed but didn't convert.
columns = ["incident_id", "agency_id", "incident_date", "incident_hour"]

null_compare = con.sql(f"""
    WITH raw AS (
        SELECT source_year,
               {", ".join(f"count(*) FILTER (WHERE {c} IS NULL) AS {c}" for c in columns)}
        FROM incident
        GROUP BY source_year
    ),
    clean AS (
        SELECT source_year,
               {", ".join(f"count(*) FILTER (WHERE {c} IS NULL) AS {c}" for c in columns)}
        FROM incident_clean
        GROUP BY source_year
    )
    SELECT r.source_year,
           {", ".join(f"r.{c} AS {c}_raw, c.{c} AS {c}_clean, c.{c} - r.{c} AS {c}_failed" for c in columns)}
    FROM raw r
    JOIN clean c USING (source_year)
    ORDER BY r.source_year
""").df()
display(null_compare)

failed_cols = [f"{c}_failed" for c in columns]
total_failed = null_compare[failed_cols].sum()
print(total_failed)
assert total_failed.sum() == 0, "some values didn't convert, look at them before moving on"

# date range, and incidents whose date year doesn't match the file year
display(con.sql("""
    SELECT min(incident_date) AS first_date, max(incident_date) AS last_date
    FROM incident_clean
""").df())

display(con.sql("""
    SELECT source_year, year(incident_date) AS date_year, count(*) AS incidents
    FROM incident_clean
    WHERE year(incident_date) <> source_year
    GROUP BY ALL
    ORDER BY incidents DESC
    LIMIT 15
""").df())


,source_year,incident_id_raw,incident_id_clean,incident_id_failed,agency_id_raw,agency_id_clean,agency_id_failed,incident_date_raw,incident_date_clean,incident_date_failed,incident_hour_raw,incident_hour_clean,incident_hour_failed
0,2016,0,0,0,0,0,0,0,0,0,130153,130153,0
1,2017,0,0,0,0,0,0,0,0,0,93231,93231,0
2,2018,0,0,0,0,0,0,0,0,0,78026,78026,0
3,2019,0,0,0,0,0,0,0,0,0,70361,70361,0
4,2020,0,0,0,0,0,0,0,0,0,92659,92659,0
5,2021,0,0,0,0,0,0,0,0,0,102685,102685,0
6,2022,0,0,0,0,0,0,0,0,0,134046,134046,0
7,2023,0,0,0,0,0,0,0,0,0,143787,143787,0
8,2024,0,0,0,0,0,0,0,0,0,136017,136017,0
9,2025,0,0,0,0,0,0,0,0,0,171189,171189,0


incident_id_failed      0
agency_id_failed        0
incident_date_failed    0
incident_hour_failed    0
dtype: int64


,first_date,last_date
0,2016-01-01,2025-12-31


,source_year,date_year,incidents


## Clean Offense Table

(offense code -> 2021+ has it directly; 2016-2020 via offense_type_id)

(kept attempt_complete_flag, A = attempted, C = completed)

In [6]:
con.sql("""
    CREATE OR REPLACE TABLE offense_clean AS
    SELECT
        try_cast(o.offense_id AS BIGINT)        AS offense_id,
        try_cast(o.incident_id AS BIGINT)       AS incident_id,
        o.source_state,
        o.source_year,
        coalesce(o.offense_code, ot.offense_code) AS offense_code,   -- 
        o.attempt_complete_flag                 AS attempt_complete  -- 
    FROM offense o
    LEFT JOIN offense_type ot
        ON o.offense_type_id = ot.offense_type_id
""")

raw_rows = con.sql("SELECT count(*) FROM offense").fetchall()[0][0]
clean_rows = con.sql("SELECT count(*) FROM offense_clean").fetchall()[0][0]
assert raw_rows == clean_rows, f"offense: raw {raw_rows}, clean {clean_rows}"
print(f"offense_clean: {clean_rows} rows")


offense_clean: 73794315 rows


## Verify

In [7]:
# 1. NULLs before vs after: IDs should not fail; offense_code should have no NULLs at all
display(con.sql("""
    SELECT o.source_year,
           count(*) FILTER (WHERE o.offense_id IS NULL)                         AS offense_id_raw,
           count(*) FILTER (WHERE c.offense_id IS NULL)                         AS offense_id_clean,
           count(*) FILTER (WHERE o.incident_id IS NULL)                        AS incident_id_raw,
           count(*) FILTER (WHERE c.incident_id IS NULL)                        AS incident_id_clean,
           count(*) FILTER (WHERE o.offense_code IS NULL AND o.offense_type_id IS NULL) AS no_code_raw,
           count(*) FILTER (WHERE c.offense_code IS NULL)                       AS no_code_clean
    FROM offense o
    JOIN offense_clean c
      ON try_cast(o.offense_id AS BIGINT) = c.offense_id
     AND o.source_state = c.source_state AND o.source_year = c.source_year
    GROUP BY o.source_year
    ORDER BY o.source_year
""").df())

missing = con.sql("SELECT count(*) FROM offense_clean WHERE offense_code IS NULL").fetchall()[0][0]
assert missing == 0, f"{missing} offenses have no offense_code"

# 2. the three assault codes by year: watch for a jump at 2021 (would mean the old-era mapping is off)
display(con.sql("""
    SELECT source_year,
           count(*) FILTER (WHERE offense_code = '13A') AS agg_assault_13A,
           count(*) FILTER (WHERE offense_code = '13B') AS simple_assault_13B,
           count(*) FILTER (WHERE offense_code = '13C') AS intimidation_13C,
           count(*) AS all_offenses
    FROM offense_clean
    GROUP BY source_year
    ORDER BY source_year
""").df())


,source_year,offense_id_raw,offense_id_clean,incident_id_raw,incident_id_clean,no_code_raw,no_code_clean
0,2016,0,0,0,0,0,0
1,2017,0,0,0,0,0,0
2,2018,0,0,0,0,0,0
3,2019,0,0,0,0,0,0
4,2020,0,0,0,0,0,0
5,2021,0,0,0,0,0,0
6,2022,0,0,0,0,0,0
7,2023,0,0,0,0,0,0
8,2024,0,0,0,0,0,0
9,2025,0,0,0,0,0,0


,source_year,agg_assault_13A,simple_assault_13B,intimidation_13C,all_offenses
0,2016,132043,444022,144962,3676581
1,2017,141511,472251,141870,3806492
2,2018,160874,536034,163684,4130834
3,2019,201501,669859,199675,5083657
4,2020,279156,808434,236947,6215201
5,2021,362974,1101528,311638,8011266
6,2022,433900,1357425,361783,9888831
7,2023,497338,1554209,505389,11169806
8,2024,518223,1638143,533519,11143949
9,2025,499484,1639707,549444,10667698


## Clean & Verify Offender and Victim Tables

(age currently excluded from both)

In [8]:
def null_check(raw_table, clean_table, id_col, columns):
    """NULLs per column and year, raw vs clean. failed = values that existed but didn't convert."""
    selects = ",\n".join(
        f"count(*) FILTER (WHERE r.{c} IS NULL) AS {c}_raw, "
        f"count(*) FILTER (WHERE c.{c} IS NULL) AS {c}_clean, "
        f"count(*) FILTER (WHERE c.{c} IS NULL) - count(*) FILTER (WHERE r.{c} IS NULL) AS {c}_failed"
        for c in columns
    )
    df = con.sql(f"""
        SELECT r.source_year, {selects}
        FROM {raw_table} r
        JOIN {clean_table} c
          ON try_cast(r.{id_col} AS BIGINT) = c.{id_col}
         AND r.source_state = c.source_state AND r.source_year = c.source_year
        GROUP BY r.source_year
        ORDER BY r.source_year
    """).df()
    display(df)
    failed = df[[f"{c}_failed" for c in columns]].sum()
    assert failed.sum() == 0, f"{clean_table}: values didn't convert:\n{failed[failed > 0]}"
    print(f"{clean_table}: no failed conversions")


In [9]:
con.sql("""
    CREATE OR REPLACE TABLE offender_clean AS
    SELECT
        try_cast(offender_id AS BIGINT)  AS offender_id,
        try_cast(incident_id AS BIGINT)  AS incident_id,
        source_state,
        source_year,
        sex_code
    FROM offender
""")

raw_rows = con.sql("SELECT count(*) FROM offender").fetchall()[0][0]
clean_rows = con.sql("SELECT count(*) FROM offender_clean").fetchall()[0][0]
assert raw_rows == clean_rows, f"offender: raw {raw_rows}, clean {clean_rows}"
print(f"offender_clean: {clean_rows} rows")

null_check("offender", "offender_clean", "offender_id", ["offender_id", "incident_id", "sex_code"])

display(con.sql("""
    SELECT source_year, sex_code, count(*) AS offenders
    FROM offender_clean
    GROUP BY ALL
    ORDER BY source_year, sex_code
""").df())


offender_clean: 73654163 rows


,source_year,offender_id_raw,offender_id_clean,offender_id_failed,incident_id_raw,incident_id_clean,incident_id_failed,sex_code_raw,sex_code_clean,sex_code_failed
0,2016,0,0,0,0,0,0,1025670,1025670,0
1,2017,0,0,0,0,0,0,980356,980356,0
2,2018,0,0,0,0,0,0,974980,974980,0
3,2019,0,0,0,0,0,0,1243263,1243263,0
4,2020,0,0,0,0,0,0,1697069,1697069,0
5,2021,0,0,0,0,0,0,0,0,0
6,2022,0,0,0,0,0,0,0,0,0
7,2023,0,0,0,0,0,0,0,0,0
8,2024,0,0,0,0,0,0,0,0,0
9,2025,0,0,0,0,0,0,0,0,0


offender_clean: no failed conversions


,source_year,sex_code,offenders
0,2016,F,680528
1,2016,M,1670907
2,2016,U,362633
3,2016,NaN,1025670
4,2017,F,718632
5,2017,M,1746291
6,2017,U,416875
7,2017,NaN,980356
8,2018,F,802955
9,2018,M,1941279


In [10]:
con.sql("""
    CREATE OR REPLACE TABLE victim_clean AS
    SELECT
        try_cast(victim_id AS BIGINT)       AS victim_id,
        try_cast(incident_id AS BIGINT)     AS incident_id,
        source_state,
        source_year,
        try_cast(victim_type_id AS INTEGER) AS victim_type_id,   -- 4 = individual
        sex_code
    FROM victim
""")

raw_rows = con.sql("SELECT count(*) FROM victim").fetchall()[0][0]
clean_rows = con.sql("SELECT count(*) FROM victim_clean").fetchall()[0][0]
assert raw_rows == clean_rows, f"victim: raw {raw_rows}, clean {clean_rows}"
print(f"victim_clean: {clean_rows} rows")

null_check("victim", "victim_clean", "victim_id", ["victim_id", "incident_id", "victim_type_id", "sex_code"])

display(con.sql("""
    SELECT source_year, victim_type_id, sex_code, count(*) AS victims
    FROM victim_clean
    GROUP BY ALL
    ORDER BY source_year, victim_type_id, sex_code
""").df())


victim_clean: 73955092 rows


,source_year,victim_id_raw,victim_id_clean,victim_id_failed,incident_id_raw,incident_id_clean,incident_id_failed,victim_type_id_raw,victim_type_id_clean,victim_type_id_failed,sex_code_raw,sex_code_clean,sex_code_failed
0,2016,0,0,0,0,0,0,0,0,0,1041261,1041261,0
1,2017,0,0,0,0,0,0,0,0,0,1096892,1096892,0
2,2018,0,0,0,0,0,0,0,0,0,1207525,1207525,0
3,2019,0,0,0,0,0,0,0,0,0,1478838,1478838,0
4,2020,0,0,0,0,0,0,0,0,0,1699514,1699514,0
5,2021,0,0,0,0,0,0,0,0,0,0,0,0
6,2022,0,0,0,0,0,0,0,0,0,0,0,0
7,2023,0,0,0,0,0,0,0,0,0,0,0,0
8,2024,0,0,0,0,0,0,0,0,0,0,0,0
9,2025,0,0,0,0,0,0,0,0,0,0,0,0


victim_clean: no failed conversions


,source_year,victim_type_id,sex_code,victims
0,2016,1,NaN,563103
1,2016,2,NaN,11504
2,2016,3,NaN,37340
3,2016,4,F,1363416
4,2016,4,M,1263416
...,...,...,...,...
125,2025,5,U,1813
126,2025,6,X,29527
127,2025,7,X,8698
128,2025,8,X,1304027


## Clean Victim Offender Relationship Table

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE victim_offender_rel_clean AS
    SELECT
        try_cast(victim_id AS BIGINT)        AS victim_id,
        try_cast(offender_id AS BIGINT)      AS offender_id,
        source_state,
        source_year,
        try_cast(relationship_id AS INTEGER) AS relationship_id
    FROM victim_offender_rel
""")

raw_rows = con.sql("SELECT count(*) FROM victim_offender_rel").fetchall()[0][0]
clean_rows = con.sql("SELECT count(*) FROM victim_offender_rel_clean").fetchall()[0][0]
assert raw_rows == clean_rows, f"victim_offender_rel: raw {raw_rows}, clean {clean_rows}"
print(f"victim_offender_rel_clean: {clean_rows} rows")

# no single id column, so compare NULL counts directly
display(con.sql("""
    SELECT r.source_year,
           r.n_null_victim, c.n_null_victim AS n_null_victim_clean,
           r.n_null_offender, c.n_null_offender AS n_null_offender_clean,
           r.n_null_rel, c.n_null_rel AS n_null_rel_clean
    FROM (SELECT source_year,
                 count(*) FILTER (WHERE victim_id IS NULL)       AS n_null_victim,
                 count(*) FILTER (WHERE offender_id IS NULL)     AS n_null_offender,
                 count(*) FILTER (WHERE relationship_id IS NULL) AS n_null_rel
          FROM victim_offender_rel GROUP BY 1) r
    JOIN (SELECT source_year,
                 count(*) FILTER (WHERE victim_id IS NULL)       AS n_null_victim,
                 count(*) FILTER (WHERE offender_id IS NULL)     AS n_null_offender,
                 count(*) FILTER (WHERE relationship_id IS NULL) AS n_null_rel
          FROM victim_offender_rel_clean GROUP BY 1) c USING (source_year)
    ORDER BY r.source_year
""").df())


victim_offender_rel_clean: 33158338 rows


,source_year,n_null_victim,n_null_victim_clean,n_null_offender,n_null_offender_clean,n_null_rel,n_null_rel_clean
0,2016,0,0,0,0,0,0
1,2017,0,0,0,0,0,0
2,2018,0,0,0,0,0,0
3,2019,0,0,0,0,0,0
4,2020,0,0,0,0,0,0
5,2021,0,0,0,0,0,0
6,2022,0,0,0,0,0,0
7,2023,0,0,0,0,0,0
8,2024,0,0,0,0,0,0
9,2025,0,0,0,0,0,0


## Clean Victim Offense Table

In [12]:
con.sql("""
    CREATE OR REPLACE TABLE victim_offense_clean AS
    SELECT
        try_cast(victim_id AS BIGINT)   AS victim_id,
        try_cast(offense_id AS BIGINT)  AS offense_id,
        source_state,
        source_year
    FROM victim_offense
""")

raw_rows = con.sql("SELECT count(*) FROM victim_offense").fetchall()[0][0]
clean_rows = con.sql("SELECT count(*) FROM victim_offense_clean").fetchall()[0][0]
assert raw_rows == clean_rows, f"victim_offense: raw {raw_rows}, clean {clean_rows}"
print(f"victim_offense_clean: {clean_rows} rows")

nulls = con.sql("""
    SELECT count(*) FILTER (WHERE victim_id IS NULL), count(*) FILTER (WHERE offense_id IS NULL)
    FROM victim_offense_clean
""").fetchall()[0]
assert nulls == (0, 0), f"victim_offense_clean NULL ids (victim, offense): {nulls}"
print("victim_offense_clean: no NULL ids")


victim_offense_clean: 80520599 rows
victim_offense_clean: no NULL ids


## Final NIBRS Table For Causal Analysis

In [4]:
# Create a table with the cleaned data to only include incidents with a male offender, a female victim, the type of offense being either simple assault, aggravated assault, or intimidation, 
# and the relationship between the victim and offender being either spouse, common-law spouse, boyfriend/girlfriend, ex-boyfriend/ex-girlfriend, or ex-spouse. 
# This will be used for the causal inference analysis.
cleaned_data = con.sql("""
    CREATE OR REPLACE TABLE cleaned_data AS
       SELECT i.incident_id, i.source_state, i.source_year, i.incident_date, i.incident_hour,
                 o.offense_code,
                 of.sex_code AS offender_sex,
                 v.sex_code AS victim_sex,
                 vor.relationship_id
       FROM incident_clean i
       JOIN offender_clean of ON i.incident_id = of.incident_id AND i.source_state = of.source_state AND i.source_year = of.source_year
       JOIN victim_clean v ON i.incident_id = v.incident_id AND i.source_state = v.source_state AND i.source_year = v.source_year
       JOIN offense_clean o ON i.incident_id = o.incident_id AND i.source_state = o.source_state AND i.source_year = o.source_year
       JOIN victim_offense_clean vo ON v.victim_id = vo.victim_id AND o.offense_id = vo.offense_id AND i.source_state = vo.source_state AND i.source_year = vo.source_year
       JOIN victim_offender_rel_clean vor ON v.victim_id = vor.victim_id AND of.offender_id = vor.offender_id AND i.source_state = vor.source_state AND i.source_year = vor.source_year
       WHERE of.sex_code = 'M' AND v.sex_code = 'F'
         AND o.offense_code IN ('13A', '13B', '13C')
         AND vor.relationship_id IN (3, 6, 21, 26, 27)
""") 

# 13A = Aggravated Assault, 13B = Simple Assault, 13C = Intimidation
# 3 = Boyfriend/Girlfriend, 6 = Common-Law Spouse, 21 = Spouse, 26 = Ex-Spouse, 27 = Ex-Boyfriend/Ex-Girlfriend

rows = con.sql("SELECT count(*) FROM cleaned_data").fetchall()[0][0]
print(f"cleaned_data: {rows} rows")

cleaned_data: 4058474 rows


## Count of Incidents Each Year for Each State that Hasn't Legalized Online Sports Betting

In [4]:
# Find the number of incidents per year for states that haven't legalized online sports betting
incidents_per_year = con.sql("""
    SELECT source_year, source_state, count(DISTINCT incident_id) AS incidents
    FROM cleaned_data
    WHERE source_state IN ('CA', 'GA', 'MN', 'SC', 'TX', 'WA', 'WI')
    GROUP BY source_year, source_state
    ORDER BY source_year
""").df()

# Pivot table for easier readability
incidents_per_year_pivot = incidents_per_year.pivot(index='source_year', columns='source_state', values='incidents')

incidents_per_year_pivot

source_state,CA,GA,MN,SC,TX,WA,WI
source_year,,,,,,,
2016,NaN,NaN,NaN,22947.0,12789.0,14616.0,5946.0
2017,NaN,NaN,NaN,22412.0,17153.0,15303.0,9008.0
2018,NaN,26.0,906.0,22920.0,37443.0,15902.0,9810.0
2019,NaN,4234.0,4173.0,22451.0,54949.0,14745.0,10740.0
2020,NaN,19357.0,7066.0,22649.0,80549.0,14644.0,13174.0
2021,370.0,23372.0,10181.0,21653.0,107299.0,14229.0,13453.0
2022,32514.0,23530.0,9515.0,20927.0,101430.0,14454.0,12785.0
2023,54364.0,22879.0,9483.0,20823.0,97974.0,14459.0,12555.0
2024,67100.0,22907.0,10134.0,19834.0,98405.0,14339.0,12430.0


## Clean Games Table

(does not count ties as upsets, includes playoff games as of now)

In [13]:
# spread_line > 0 means the HOME team was favored (checked: home favorites of 4+ win 75%).
# result = home_score - away_score.
con.sql("""
    CREATE OR REPLACE TABLE games_clean AS
    SELECT
        game_id, season, game_type, week, gameday, location,
        home_team, away_team, home_score, away_score, result, spread_line,
        CASE WHEN spread_line > 0 THEN home_team
             WHEN spread_line < 0 THEN away_team END          AS favorite,
        CASE WHEN spread_line > 0 THEN away_team
             WHEN spread_line < 0 THEN home_team END          AS underdog,
        abs(spread_line)                                      AS fav_spread,
        CASE WHEN spread_line > 0 THEN result < 0
             WHEN spread_line < 0 THEN result > 0 END         AS favorite_lost,   -- ties count as not lost
        coalesce(abs(spread_line) >= 4 AND
                 CASE WHEN spread_line > 0 THEN result < 0
                      WHEN spread_line < 0 THEN result > 0 END, false) AS upset
    FROM games
    WHERE season BETWEEN 2016 AND 2025
      AND result IS NOT NULL
""")

display(con.sql("""
    SELECT season,
           count(*) AS games,
           count(*) FILTER (WHERE fav_spread >= 4) AS fav_by_4_plus,
           count(*) FILTER (WHERE upset) AS upsets,
           round(100.0 * count(*) FILTER (WHERE upset) / count(*) FILTER (WHERE fav_spread >= 4), 1) AS upset_pct
    FROM games_clean
    GROUP BY season
    ORDER BY season
""").df())


,season,games,fav_by_4_plus,upsets,upset_pct
0,2016,267,124,35,28.2
1,2017,267,145,32,22.1
2,2018,267,143,35,24.5
3,2019,267,156,40,25.6
4,2020,269,154,33,21.4
5,2021,285,169,45,26.6
6,2022,284,139,34,24.5
7,2023,285,131,41,31.3
8,2024,285,148,32,21.6
9,2025,285,143,36,25.2


## Load Team States Table

In [3]:
STATES = (
    "AZ CA CO DC FL GA IL IN KS LA MD MA MI MN MO "
    "NV NJ NY NC OH PA SC TN TX WA WI"
).split()

team_states = pd.DataFrame([
    ("ARI", "AZ"), ("ATL", "GA"), ("BAL", "MD"), ("BUF", "NY"),
    ("CAR", "NC"), ("CAR", "SC"),
    ("CHI", "IL"), ("CIN", "OH"), ("CLE", "OH"), ("DAL", "TX"), ("DEN", "CO"),
    ("DET", "MI"), ("GB", "WI"), ("HOU", "TX"), ("IND", "IN"), ("JAX", "FL"),
    ("KC", "MO"), ("KC", "KS"),
    ("LA", "CA"), ("LAC", "CA"), ("SD", "CA"),      # SD = Chargers 2016, LAC = 2017+
    ("OAK", "CA"), ("LV", "NV"),                    # OAK = Raiders 2016-2019, LV = 2020+
    ("MIA", "FL"), ("MIN", "MN"), ("NE", "MA"), ("NO", "LA"),
    ("NYG", "NY"), ("NYG", "NJ"), ("NYJ", "NY"), ("NYJ", "NJ"),
    ("PHI", "PA"), ("PIT", "PA"), ("SEA", "WA"), ("SF", "CA"), ("TB", "FL"),
    ("TEN", "TN"),
    ("WAS", "MD"), ("WAS", "DC"),
], columns=["team", "state"])

con.sql("CREATE OR REPLACE TABLE team_states AS SELECT * FROM team_states")

# every team in games_clean has a state; every state in STATES has a team
unmapped = con.sql("""
    SELECT DISTINCT t FROM (SELECT home_team AS t FROM games_clean UNION SELECT away_team FROM games_clean)
    WHERE t NOT IN (SELECT team FROM team_states)
""").fetchall()
assert unmapped == [], f"teams with no state: {unmapped}"
no_team = sorted(set(STATES) - set(team_states["state"]))
assert no_team == [], f"states with no team: {no_team}"
print(f"team_states: {len(team_states)} rows, all teams and states covered")


team_states: 39 rows, all teams and states covered


## Scope States

In [5]:
scope_states = pd.DataFrame([
    # (state, group from SCOPE.md, legal online betting launch date: FILL IN, 'YYYY-MM-DD' or None)
    ("CA", "not legal", None), ("GA", "not legal", None), ("MN", "not legal", None),
    # Missouri did legalize online sports betting, but on December 1, 2025, 
    # so we will just use None for our analysis.
    ("MO", "not legal", None), ("SC", "not legal", None), 
    ("TX", "not legal", None), ("WA", "not legal", None), ("WI", "not legal", None),
    ("AZ", "legal", '2021-09-09'), ("CO", "legal", '2020-05-01'), ("FL", "legal", '2023-11-07'),
    ("IL", "legal", '2020-06-18'), ("IN", "legal", '2019-10-03'), ("KS", "legal", '2022-09-08'),
    ("LA", "legal", '2022-01-28'), ("MD", "legal", '2022-11-23'), ("MA", "legal", '2023-03-10'),
    # Nevada has long had legal online sports betting,
    # but the first year used for our analysis is 2016, 
    # so we will just use January 1, 2016 as the legal date for our analysis.
    ("MI", "legal", '2021-01-22'), ("NV", "legal", '2016-01-01'),
    ("NJ", "legal", '2018-08-01'), ("NY", "legal", '2022-01-08'), ("NC", "legal", '2024-03-11'),
    ("OH", "legal", '2023-01-01'), ("PA", "legal", '2019-05-28'), ("TN", "legal", '2020-11-01'),
    ("DC", "legal", '2020-05-28'),
], columns=["state", "scope_group", "legal_date"])

con.sql("""
    CREATE OR REPLACE TABLE scope_states AS
    SELECT state, scope_group, try_cast(legal_date AS DATE) AS legal_date
    FROM scope_states
""")

assert sorted(scope_states["state"]) == sorted(STATES), "scope_states and STATES don't match"
missing_dates = scope_states.query("scope_group == 'legal' and legal_date.isna()", engine="python")["state"].tolist()
print(f"scope_states: {len(scope_states)} states; legal states still missing a date: {missing_dates}")


scope_states: 26 states; legal states still missing a date: []


## Final Verifications

In [6]:
clean_joins = [
    # (child, column, parent, column)
    ("offense_clean", "incident_id", "incident_clean", "incident_id"),
    ("offender_clean", "incident_id", "incident_clean", "incident_id"),
    ("victim_clean", "incident_id", "incident_clean", "incident_id"),
    ("victim_offense_clean", "victim_id", "victim_clean", "victim_id"),
    ("victim_offense_clean", "offense_id", "offense_clean", "offense_id"),
    ("victim_offender_rel_clean", "victim_id", "victim_clean", "victim_id"),
    ("victim_offender_rel_clean", "offender_id", "offender_clean", "offender_id"),
]
known_gaps = {
    ("offender_clean", "incident_id"): "2017 offenders whose incident is not in the incident file",
    ("victim_offender_rel_clean", "offender_id"): "2017 relationships whose offender is not in the offender file",
}

for child, child_col, parent, parent_col in clean_joins:
    orphans = con.sql(f"""
        SELECT count(*)
        FROM {child} c
        LEFT JOIN {parent} p
          ON c.{child_col} = p.{parent_col}
         AND c.source_state = p.source_state AND c.source_year = p.source_year
        WHERE c.{child_col} IS NOT NULL AND p.{parent_col} IS NULL
    """).fetchall()[0][0]
    if orphans and (child, child_col) in known_gaps:
        print(f"{child}.{child_col} -> {parent}: {orphans} rows, known gap: {known_gaps[(child, child_col)]}")
        continue
    assert orphans == 0, f"{child}.{child_col}: {orphans} rows not found in {parent}"
    print(f"{child}.{child_col} -> {parent}: ok")


offense_clean.incident_id -> incident_clean: ok
offender_clean.incident_id -> incident_clean: 17207 rows, known gap: 2017 offenders whose incident is not in the incident file
victim_clean.incident_id -> incident_clean: ok
victim_offense_clean.victim_id -> victim_clean: ok
victim_offense_clean.offense_id -> offense_clean: ok
victim_offender_rel_clean.victim_id -> victim_clean: ok
victim_offender_rel_clean.offender_id -> offender_clean: 5003 rows, known gap: 2017 relationships whose offender is not in the offender file
